In [2]:
import wandb
import numpy as np
import matplotlib.pyplot as plt

project_path = "pippoborsa/Isaac-robot-US-guidance-G1-v0"  
group_name   = "MB-US-sim"                           

title = "US guidance G1"  

x_key = "_step"        # o "episode", "global_step", ecc. → controlla in history.columns
y_key = "total_reward" 

In [3]:
def download_group_runs(project_path, group_name=None):
    """Download all runs from a given project (optionally filtered by group)."""
    api = wandb.Api()

    if group_name is not None:
        runs = api.runs(project_path, filters={"group": group_name})
    else:
        runs = api.runs(project_path)

    all_curves = []

    for run in runs:
        try:
            # Per debug: guarda le colonne disponibili
            history = run.history()
            print(run.name, history.columns.tolist())

            df = run.history(keys=[x_key, y_key])

            # Keep only rows where both x and y are not NaN
            df = df[[x_key, y_key]].dropna()

            # Rename for easier handling
            df = df.rename(columns={x_key: "x", y_key: "y"})
            all_curves.append(df)
        except Exception as e:
            print(f"Failed to load run {run.name}: {e}")

    return all_curves

In [4]:
def align_curves(curves):
    """Align multiple curves by truncating all to the minimum length.
    
    curves: list of DataFrame, each with columns ["x", "y"].
    Returns:
        x_common: np.ndarray of shape (T,)
        y_all:   np.ndarray of shape (N_runs, T)
    """
    if len(curves) == 0:
        raise ValueError("No curves to align.")

    # Minimum length among all runs
    min_len = min(len(df) for df in curves)

    # Use x from the first run as reference (truncated)
    x_common = curves[0]["x"].values[:min_len]

    # Stack y values from all runs (truncated)
    y_all = np.stack([df["y"].values[:min_len] for df in curves], axis=0)  # (N_runs, T)

    return x_common, y_all

In [ ]:
# Download runs for this agent (KUKA or G1)
curves = download_group_runs(project_path, group_name)

if len(curves) == 0:
    raise RuntimeError("No runs found for the given project/group. Check names and filters.")

# Align curves and compute statistics
x, y_all = align_curves(curves)       # x: (T,), y_all: (N_runs, T)
mean_y = y_all.mean(axis=0)          # (T,)
std_y  = y_all.std(axis=0)           # (T,)

# --- Plotting ---
plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 12,
    "figure.titlesize": 18,
})

plt.figure(figsize=(6, 4))

plt.plot(x, mean_y, label="policy", color="cornflowerblue", linewidth=2)
plt.fill_between(x, mean_y - std_y, mean_y + std_y,
                 alpha=0.3, color="cornflowerblue", label="±1 std")

plt.xlabel(x_key)          # es. "episodes" se usi quello
plt.ylabel(y_key)          # es. "total_reward"
plt.title(title)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

wandb: WARNING Failed to create global config settings in: /Users/xsixsipas/.config/wandb. Settings will not be persisted.
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter: